# Basic Backtesting with kimsfinance

This notebook demonstrates how to:
1. Load and prepare OHLCV data
2. Create a simple RSI strategy
3. Run a backtest with Rust acceleration
4. Visualize and analyze results

## Prerequisites
```bash
cd rust/
maturin develop --release
pip install matplotlib pandas
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import kimsfinance_core
from kimsfinance.strategies import RSIStrategy
from kimsfinance.visualization import plot_equity_curve, plot_drawdown, plot_performance_dashboard, print_performance_summary

print(f"kimsfinance_core v{kimsfinance_core.__version__}")

## 1. Generate Sample Data

For this example, we'll generate synthetic OHLCV data. In practice, you would load real market data.

In [ ]:
def generate_sample_data(n=1000, trend='up', seed=42):
    """
    Generate synthetic OHLCV data for testing
    
    Parameters:
    - n: Number of candles
    - trend: 'up', 'down', or 'sideways'
    - seed: Random seed for reproducibility
    """
    np.random.seed(seed)
    
    timestamps = np.arange(n, dtype=np.int64) * 60  # 1-minute bars
    
    if trend == 'up':
        base = np.linspace(100.0, 200.0, n)
        noise = np.random.randn(n).cumsum() * 2
        close = base + noise
    elif trend == 'down':
        base = np.linspace(200.0, 100.0, n)
        noise = np.random.randn(n).cumsum() * 2
        close = base + noise
    else:  # sideways
        base = 150.0 + 20 * np.sin(np.arange(n) / 20)
        noise = np.random.randn(n) * 5
        close = base + noise
    
    # Generate realistic OHLC from close
    open_prices = close + np.random.randn(n) * 0.5
    high = np.maximum(open_prices, close) + np.abs(np.random.randn(n) * 2)
    low = np.minimum(open_prices, close) - np.abs(np.random.randn(n) * 2)
    volume = np.random.uniform(1000, 10000, n)
    
    return timestamps, open_prices, high, low, close, volume

# Generate data
timestamps, open_p, high, low, close, volume = generate_sample_data(1000, trend='up')

print(f"Generated {len(close)} candles")
print(f"Price range: ${low.min():.2f} - ${high.max():.2f}")
print(f"Average volume: {volume.mean():.0f}")

## 2. Visualize the Data

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Price chart
ax1.plot(close, color='#2E86AB', linewidth=1.5, label='Close Price')
ax1.fill_between(range(len(high)), low, high, alpha=0.2, color='#2E86AB')
ax1.set_title('Price Chart', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price ($)', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Volume
ax2.bar(range(len(volume)), volume, color='#F18F01', alpha=0.6)
ax2.set_title('Volume', fontsize=14, fontweight='bold')
ax2.set_xlabel('Candles', fontsize=12)
ax2.set_ylabel('Volume', fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Create a Simple RSI Strategy

We'll use a classic RSI mean reversion strategy:
- Buy when RSI < 30 (oversold)
- Sell when RSI > 70 (overbought)

In [ ]:
# Create strategy instance
strategy = RSIStrategy(
    period=14,
    buy_threshold=30,
    sell_threshold=70,
    position_pct=1.0  # 100% allocation
)

print("Strategy: RSI Mean Reversion")
print(f"  Period: {strategy.period}")
print(f"  Buy threshold: {strategy.buy_threshold}")
print(f"  Sell threshold: {strategy.sell_threshold}")
print(f"  Required indicators: {strategy.get_indicators()}")

## 4. Run the Backtest

Now let's run the backtest using the Rust-accelerated engine.

In [ ]:
result = kimsfinance_core.run_backtest(
    high=high,
    low=low,
    close=close,
    open_prices=open_p,
    volume=volume,
    timestamps=timestamps,
    strategy=strategy,
    initial_capital=10000.0,
    trading_fee=0.001,    # 0.1% per trade
    slippage=0.0005,      # 0.05% slippage
    use_gpu=False         # CPU mode (set True if GPU available)
)

print("Backtest completed!")
print_performance_summary(result)

## 5. Visualize Results

### Equity Curve

In [ ]:
fig = plot_equity_curve(result, title="RSI Strategy Equity Curve")
plt.show()

### Drawdown

In [ ]:
fig = plot_drawdown(result)
plt.show()

### Complete Performance Dashboard

In [ ]:
fig = plot_performance_dashboard(result)
plt.show()

## 6. Analyze Trades

Let's examine individual trades to understand the strategy behavior.

In [ ]:
trades_df = pd.DataFrame(result['trades'])
print(f"Total trades: {len(trades_df)}")
print(f"\nTrade statistics:")
print(trades_df[['pnl', 'pnl_percent']].describe())

# Show first 10 trades
print("\nFirst 10 trades:")
trades_df[['entry_time', 'exit_time', 'entry_price', 'exit_price', 'pnl', 'pnl_percent']].head(10)

## 7. Calculate RSI and Overlay on Price Chart

Let's visualize the RSI indicator to see when buy/sell signals occurred.

In [ ]:
# Calculate RSI using kimsfinance_core
rsi = kimsfinance_core.calculate_rsi(close, period=14)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Price chart with trades
ax1.plot(close, color='#2E86AB', linewidth=1.5, label='Close Price')

# Mark trade entries and exits
for trade in result['trades'][:20]:  # Show first 20 trades
    entry_idx = np.searchsorted(timestamps, trade['entry_time'])
    exit_idx = np.searchsorted(timestamps, trade['exit_time'])
    
    if trade['direction'] == 'long':
        ax1.scatter(entry_idx, trade['entry_price'], color='green', marker='^', s=100, zorder=5)
        ax1.scatter(exit_idx, trade['exit_price'], color='red', marker='v', s=100, zorder=5)

ax1.set_title('Price Chart with Trade Signals', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price ($)', fontsize=12)
ax1.legend(['Close', 'Buy', 'Sell'])
ax1.grid(True, alpha=0.3)

# RSI
ax2.plot(rsi, color='purple', linewidth=1.5)
ax2.axhline(y=70, color='red', linestyle='--', alpha=0.7, label='Overbought (70)')
ax2.axhline(y=30, color='green', linestyle='--', alpha=0.7, label='Oversold (30)')
ax2.fill_between(range(len(rsi)), 0, 30, color='green', alpha=0.1)
ax2.fill_between(range(len(rsi)), 70, 100, color='red', alpha=0.1)
ax2.set_title('RSI(14)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Candles', fontsize=12)
ax2.set_ylabel('RSI', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 100)

plt.tight_layout()
plt.show()

## 8. Next Steps

Now that you've run a basic backtest, try:

1. **Parameter Optimization**: See `02_parameter_optimization.ipynb` to optimize RSI thresholds
2. **Different Strategies**: Try trend-following or volatility strategies
3. **Real Data**: Load actual market data from CSV or API
4. **Multiple Indicators**: Combine RSI with other indicators for better signals

## Performance Notes

- This backtest runs on CPU for compatibility
- GPU acceleration available with `use_gpu=True` (requires CUDA-capable GPU)
- Expected speedup: 10-50x vs pure Python implementations
- Indicator calculations are Rust-accelerated (5-10x faster than pandas)